# Phase 3A — CNN Spatial Feature Extraction

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Define Paths

In [ ]:
import os

# Root project directory
DRIVE_ROOT = "/content/drive/MyDrive/FYP_YOLO_Hand_Detection"

# ROI image folders
ROI_ROOT = f"{DRIVE_ROOT}/roi_images_v3"

# Output CNN feature folder
FEATURE_ROOT = f"{DRIVE_ROOT}/cnn_features"

splits = ["train", "valid", "test"]

for split in splits:
    os.makedirs(os.path.join(FEATURE_ROOT, split), exist_ok=True)

print("Directories ready.")

Directories ready.


## Install & Import Dependencies

In [ ]:
!pip install torch torchvision tqdm

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

## Load Pretrained CNN (ResNet50)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load ResNet50
resnet = models.resnet50(pretrained=True)

# Remove classification layer
resnet = nn.Sequential(*list(resnet.children())[:-1])

resnet = resnet.to(device)
resnet.eval()

print("ResNet50 loaded as feature extractor")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 388MB/s]


ResNet50 loaded as feature extractor


## Image Preprocessing Pipeline

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Feature Extraction Function

In [ ]:
def extract_features(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = resnet(image)

    return features.squeeze().cpu().numpy()

## Extract Features for All Splits

In [ ]:
metadata = []

for split in splits:
    print(f"\nProcessing {split.upper()} set")

    split_input = os.path.join(ROI_ROOT, split)
    split_output = os.path.join(FEATURE_ROOT, split)

    for class_name in os.listdir(split_input):
        class_input = os.path.join(split_input, class_name)
        class_output = os.path.join(split_output, class_name)

        if not os.path.isdir(class_input):
            continue

        os.makedirs(class_output, exist_ok=True)

        images = [f for f in os.listdir(class_input) if f.endswith(".jpg")]

        for img_name in tqdm(images, desc=f"{split}/{class_name}"):
            img_path = os.path.join(class_input, img_name)

            features = extract_features(img_path)

            feature_name = img_name.replace(".jpg", ".npy")
            feature_path = os.path.join(class_output, feature_name)

            np.save(feature_path, features)

            metadata.append({
                "split": split,
                "class": class_name,
                "image": img_name,
                "feature_file": feature_path
            })


Processing TRAIN set


train/Block1: 100%|██████████| 10/10 [00:07<00:00,  1.36it/s]



Processing VALID set


valid/AudiBlock: 100%|██████████| 2/2 [00:01<00:00,  1.31it/s]



Processing TEST set


test/Block2: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


## Save Feature Metadata CSV

In [ ]:
df = pd.DataFrame(metadata)

csv_path = os.path.join(FEATURE_ROOT, "feature_metadata.csv")
df.to_csv(csv_path, index=False)

print("Feature extraction complete!")
print("Metadata saved at:", csv_path)
print("Total feature vectors:", len(df))

Feature extraction complete!
Metadata saved at: /content/drive/MyDrive/FYP_YOLO_Hand_Detection/cnn_features/feature_metadata.csv
Total feature vectors: 412


# Phase 3B: Temporal Sequence Construction (CNN → BiLSTM Input)

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Define Paths

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

SEQUENCE_ROOT = "/content/drive/MyDrive/FYP_YOLO_Hand_Detection/sequence_data"
FEATURE_ROOT = "/content/drive/MyDrive/FYP_YOLO_Hand_Detection/cnn_features"

os.makedirs(SEQUENCE_ROOT, exist_ok=True)

## Load Metadata

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/FYP_YOLO_Hand_Detection/cnn_features/feature_metadata.csv"
)

print(df.columns)
df.head()

Index(['split', 'class', 'image', 'feature_file'], dtype='object')


,split,class,image,feature_file
0,train,Registrar,Registrar_2_mp4-0013_jpg.rf.0315785ef4a8f82703...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...
1,train,Registrar,Registrar_4_mp4-0004_jpg.rf.e1145885c3d23c4825...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...
2,train,Registrar,Registrar_5_mp4-0013_jpg.rf.901f0204f89e3ed014...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...
3,train,Registrar,Registrar_6_mp4-0015_jpg.rf.4c4354685d9c0aeb05...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...
4,train,Registrar,Registrar_4_mp4-0020_jpg.rf.8c3e0a42acced3206d...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...


## Create video_id and frame_index

In [ ]:
def extract_video_id_and_frame(path):
    filename = os.path.basename(path)

    # Remove _roi_v3.npy
    base = filename.replace("_roi_v3.npy", "")

    # Split frame index
    video_id, frame_part = base.rsplit("-", 1)
    frame_index = int(frame_part.split("_")[0])

    return video_id, frame_index


df[["video_id", "frame_index"]] = df["feature_file"].apply(
    lambda x: pd.Series(extract_video_id_and_frame(x))
)

df.head()

,split,class,image,feature_file,video_id,frame_index
0,train,Registrar,Registrar_2_mp4-0013_jpg.rf.0315785ef4a8f82703...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_2_mp4,13
1,train,Registrar,Registrar_4_mp4-0004_jpg.rf.e1145885c3d23c4825...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_4_mp4,4
2,train,Registrar,Registrar_5_mp4-0013_jpg.rf.901f0204f89e3ed014...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_5_mp4,13
3,train,Registrar,Registrar_6_mp4-0015_jpg.rf.4c4354685d9c0aeb05...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_6_mp4,15
4,train,Registrar,Registrar_4_mp4-0020_jpg.rf.8c3e0a42acced3206d...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_4_mp4,20


## Build Temporal Sequences

Design choices (IMPORTANT for report)

One sequence = one video

Sequence length = number of frames detected by YOLO

No padding or truncation yet

Features kept in correct temporal order

In [ ]:
sequence_metadata = []

grouped = df.groupby(["split", "class", "video_id"])

for (split, class_name, video_id), group in tqdm(grouped):
    group = group.sort_values("frame_index")

    features = []
    for _, row in group.iterrows():
        features.append(np.load(row["feature_file"]))

    if len(features) < 2:
        continue  # skip extremely short videos

    sequence = np.stack(features)  # (T, 2048)

    # Create folders
    out_dir = os.path.join(SEQUENCE_ROOT, split, class_name)
    os.makedirs(out_dir, exist_ok=True)

    seq_path = os.path.join(out_dir, f"{video_id}.npy")
    np.save(seq_path, sequence)

    sequence_metadata.append({
        "split": split,
        "class": class_name,
        "video_id": video_id,
        "sequence_file": seq_path,
        "num_frames": sequence.shape[0]
    })

100%|██████████| 75/75 [00:01<00:00, 44.85it/s]


## Save sequence metadata

In [ ]:
seq_df = pd.DataFrame(sequence_metadata)

meta_path = os.path.join(SEQUENCE_ROOT, "sequence_metadata.csv")
seq_df.to_csv(meta_path, index=False)

print(f"✅ Sequences saved to Drive")
print(f"📄 Metadata saved at: {meta_path}")
print(f"🎥 Total sequences: {len(seq_df)}")

✅ Sequences saved to Drive
📄 Metadata saved at: /content/drive/MyDrive/FYP_YOLO_Hand_Detection/sequence_data/sequence_metadata.csv
🎥 Total sequences: 60


## Sanity Check One Sequence

In [ ]:
# Check one sequence
sample = seq_df.iloc[0]
seq = np.load(sample["sequence_file"])

print("Sequence shape:", seq.shape)
print("Frames:", sample["num_frames"])
print("Class:", sample["class"])


Sequence shape: (3, 2048)
Frames: 3
Class: Block3


# Phase 3C — BiLSTM Temporal Classification

## Imports & Drive Mount

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

## Paths & Load Metadata

In [ ]:
print(df.columns)
df.head()

Index(['split', 'class', 'image', 'feature_file', 'video_id', 'frame_index'], dtype='object')


,split,class,image,feature_file,video_id,frame_index
0,train,Registrar,Registrar_2_mp4-0013_jpg.rf.0315785ef4a8f82703...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_2_mp4,13
1,train,Registrar,Registrar_4_mp4-0004_jpg.rf.e1145885c3d23c4825...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_4_mp4,4
2,train,Registrar,Registrar_5_mp4-0013_jpg.rf.901f0204f89e3ed014...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_5_mp4,13
3,train,Registrar,Registrar_6_mp4-0015_jpg.rf.4c4354685d9c0aeb05...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_6_mp4,15
4,train,Registrar,Registrar_4_mp4-0020_jpg.rf.8c3e0a42acced3206d...,/content/drive/MyDrive/FYP_YOLO_Hand_Detection...,Registrar_4_mp4,20


## Build Video Sequences

In [ ]:
sequences = []
labels = []
splits = []

grouped = df.groupby("video_id")

for video_id, group in tqdm(grouped, desc="Building sequences"):
    group = group.sort_values("frame_index")

    feature_list = []
    for _, row in group.iterrows():
        feature = np.load(row["feature_file"])
        feature_list.append(feature)

    if len(feature_list) < 2:
        continue  # skip very short videos

    sequence = np.stack(feature_list)  # (T, 2048)

    sequences.append(sequence)
    labels.append(group["class"].iloc[0])
    splits.append(group["split"].iloc[0])

print("Total sequences:", len(sequences))

Building sequences: 100%|██████████| 30/30 [00:01<00:00, 22.34it/s]

Total sequences: 30


In [ ]:
from torch.nn.utils.rnn import pad_sequence

tensor_sequences = [torch.tensor(seq, dtype=torch.float32) for seq in sequences]

padded_sequences = pad_sequence(
    tensor_sequences,
    batch_first=True
)  # (N, max_T, 2048)

print("Padded shape:", padded_sequences.shape)

Padded shape: torch.Size([30, 31, 2048])


In [ ]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)

Classes: ['AudiBlock' 'Block1' 'Block2' 'Block3' 'Block4' 'Exam' 'R-DBlock'
 'Registrar' 'office']


In [ ]:
X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

for i, split in enumerate(splits):
    if split == "train":
        X_train.append(padded_sequences[i])
        y_train.append(encoded_labels[i])
    elif split == "valid":
        X_val.append(padded_sequences[i])
        y_val.append(encoded_labels[i])
    else:
        X_test.append(padded_sequences[i])
        y_test.append(encoded_labels[i])

X_train = torch.stack(X_train)
X_val   = torch.stack(X_val)
X_test  = torch.stack(X_test)

y_train = torch.tensor(y_train)
y_val   = torch.tensor(y_val)
y_test  = torch.tensor(y_test)

print("Train:", X_train.shape)
print("Valid:", X_val.shape)
print("Test :", X_test.shape)

Train: torch.Size([24, 31, 2048])
Valid: torch.Size([4, 31, 2048])
Test : torch.Size([2, 31, 2048])


In [ ]:
class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(GestureDataset(X_train, y_train), batch_size=8, shuffle=True)
val_loader   = DataLoader(GestureDataset(X_val, y_val), batch_size=8)
test_loader  = DataLoader(GestureDataset(X_test, y_test), batch_size=8)

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        hn = torch.cat((hn[-2], hn[-1]), dim=1)
        return self.fc(hn)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM(input_dim=2048, hidden_dim=256, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

patience = 5
best_val_loss = float("inf")
patience_counter = 0

In [ ]:
ROOT = "/content/drive/MyDrive/FYP_YOLO_Hand_Detection"

MODEL_DIR = f"{ROOT}/lstm_model"
os.makedirs(MODEL_DIR, exist_ok=True)

EPOCHS = 40

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            val_loss += criterion(model(Xb), yb).item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"{MODEL_DIR}/best_bilstm.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

Epoch 1 | Train Loss: 5.7987 | Val Loss: 2.0396
Epoch 2 | Train Loss: 5.5357 | Val Loss: 2.0009
Epoch 3 | Train Loss: 5.2464 | Val Loss: 1.9687
Epoch 4 | Train Loss: 4.9957 | Val Loss: 1.9384
Epoch 5 | Train Loss: 4.7189 | Val Loss: 1.9215
Epoch 6 | Train Loss: 4.4361 | Val Loss: 1.8682
Epoch 7 | Train Loss: 4.1310 | Val Loss: 1.8315
Epoch 8 | Train Loss: 3.8261 | Val Loss: 1.7725
Epoch 9 | Train Loss: 3.5276 | Val Loss: 1.7149
Epoch 10 | Train Loss: 3.2591 | Val Loss: 1.6406
Epoch 11 | Train Loss: 2.9820 | Val Loss: 1.5803
Epoch 12 | Train Loss: 2.6873 | Val Loss: 1.5192
Epoch 13 | Train Loss: 2.4462 | Val Loss: 1.4639
Epoch 14 | Train Loss: 2.2136 | Val Loss: 1.3852
Epoch 15 | Train Loss: 1.9786 | Val Loss: 1.3169
Epoch 16 | Train Loss: 1.7790 | Val Loss: 1.2578
Epoch 17 | Train Loss: 1.6075 | Val Loss: 1.2003
Epoch 18 | Train Loss: 1.4220 | Val Loss: 1.1213
Epoch 19 | Train Loss: 1.2786 | Val Loss: 1.0626
Epoch 20 | Train Loss: 1.1450 | Val Loss: 0.9934
Epoch 21 | Train Loss: 1.0201

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Unique label indices present in test set
unique_test_labels = np.unique(y_true)

# Corresponding class names
test_class_names = label_encoder.inverse_transform(unique_test_labels)

print("Classes present in TEST set:", test_class_names)

Classes present in TEST set: ['R-DBlock']


In [ ]:
model.load_state_dict(torch.load(f"{MODEL_DIR}/best_bilstm.pt"))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        outputs = model(Xb)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        y_true.extend(yb.numpy())
        y_pred.extend(preds)

print(
    classification_report(
        y_true,
        y_pred,
        labels=unique_test_labels,
        target_names=test_class_names,
        zero_division=0
    ))

              precision    recall  f1-score   support

    R-DBlock       0.00      0.00      0.00       2.0

   micro avg       0.00      0.00      0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

